<a href="https://colab.research.google.com/github/heoconngoc/Deep_Learning/blob/main/6_Builders_Guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
import time, math, os, random
torch.manual_seed(0)

def try_gpu(i=0):
    if torch.cuda.device_count() >= i+1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

device = try_gpu()
device

device(type='cpu')

In [3]:
class BadNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)]  # ❌ Normal list

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class GoodNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2)])  # ✅ ModuleList

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

bad, good = BadNet(), GoodNet()

print("Bad params:", sum(p.numel() for p in bad.parameters()))
print("Good params:", sum(p.numel() for p in good.parameters()))
print("Bad state keys:", list(bad.state_dict().keys()))
print("Good state keys:", list(good.state_dict().keys())[:5], "...")

Bad params: 0
Good params: 58
Bad state keys: []
Good state keys: ['layers.0.weight', 'layers.0.bias', 'layers.2.weight', 'layers.2.bias'] ...


If we want to save dynamic layer list, we need to use nn.ModuleList or nn.Sequential, nn.ModuleDict

In [4]:
class ParallelConcat(nn.Module):
    def __init__(self, net1: nn.Module, net2: nn.Module, dim=1):
        super().__init__()
        self.net1 = net1
        self.net2 = net2
        self.dim = dim

    def forward(self, x):
        y1 = self.net1(x)
        y2 = self.net2(x)
        return torch.cat([y1, y2], dim=self.dim)

net1 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
net2 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 5))
par = ParallelConcat(net1, net2)

x = torch.randn(2, 4)
y = par(x)
y.shape


torch.Size([2, 8])

# End Chapter Project

In [1]:
# Setup & Import

import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
# Synthetic data

# input: 4 features
# output: regression (1 value)

X = torch.randn(100, 4).to(device) # 100 samples, each sample has 4 features
y = torch.sum(X, dim=1, keepdim=True)  # easy target: y = x1 + x2 + x3 + x4

print(X.shape, y.shape)

torch.Size([100, 4]) torch.Size([100, 1])


In [3]:
# Normal MLP

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x)

model_a = MLP().to(device)
model_a

MLP(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [4]:
# Model B with Shared Parameter

class SharedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Linear(8, 8)  # shared layer

        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            self.shared,
            nn.ReLU(),
            self.shared,  # use shared layer again
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x)

model_b = SharedMLP().to(device)
model_b

SharedMLP(
  (shared): Linear(in_features=8, out_features=8, bias=True)
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=8, bias=True)
    (5): ReLU()
    (6): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [5]:
# Inspect parameters

def inspect_model(model):
    print("=== PARAMETERS ===")
    for name, param in model.named_parameters():
        print(name, param.shape)

print("Model A:")
inspect_model(model_a)

print("\nModel B:")
inspect_model(model_b)

Model A:
=== PARAMETERS ===
net.0.weight torch.Size([8, 4])
net.0.bias torch.Size([8])
net.2.weight torch.Size([1, 8])
net.2.bias torch.Size([1])

Model B:
=== PARAMETERS ===
shared.weight torch.Size([8, 8])
shared.bias torch.Size([8])
net.0.weight torch.Size([8, 4])
net.0.bias torch.Size([8])
net.6.weight torch.Size([1, 8])
net.6.bias torch.Size([1])


In [6]:
print("ID of shared weight:", id(model_b.shared.weight))

for name, param in model_b.named_parameters():
    if "shared" in name:
        print(name, id(param))

ID of shared weight: 135890717958992
shared.weight 135890717958992
shared.bias 135890717959232


In [7]:
def train(model, X, y, epochs=5):
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

    for epoch in range(epochs):
        y_pred = model(X)
        loss = F.mse_loss(y_pred, y)

        optimizer.zero_grad()
        loss.backward()

        if hasattr(model, "shared"):
            grad_norm = model.shared.weight.grad.norm().item()
            print(f"Epoch {epoch} | Loss {loss.item():.4f} | Grad(shared) {grad_norm:.4f}")
        else:
            print(f"Epoch {epoch} | Loss {loss.item():.4f}")

        optimizer.step()

In [8]:
print("Training Model A")
train(model_a, X, y)

Training Model A
Epoch 0 | Loss 4.3893
Epoch 1 | Loss 3.6143
Epoch 2 | Loss 3.0331
Epoch 3 | Loss 2.4567
Epoch 4 | Loss 1.8593


In [9]:
print("Training Model B (Shared Layer)")
train(model_b, X, y)

Training Model B (Shared Layer)
Epoch 0 | Loss 4.6745 | Grad(shared) 1.3475
Epoch 1 | Loss 4.3274 | Grad(shared) 0.3345
Epoch 2 | Loss 4.2278 | Grad(shared) 0.2420
Epoch 3 | Loss 4.1715 | Grad(shared) 0.2142
Epoch 4 | Loss 4.1373 | Grad(shared) 0.1990


In [10]:
print("Shared weight gradient:")
print(model_b.shared.weight.grad[:2])  # xem 2 dòng đầu

Shared weight gradient:
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0057, -0.0006,  0.0000, -0.0011, -0.0014, -0.0035, -0.0067, -0.0016]])


In [11]:
torch.save(model_b.state_dict(), "model_b.pt")
print("Model saved!")

Model saved!


In [12]:
model_loaded = SharedMLP().to(device)
model_loaded.load_state_dict(torch.load("model_b.pt"))

print("Model loaded!")

Model loaded!


In [14]:
x_test = torch.randn(5, 4).to(device)

y1 = model_b(x_test)
y2 = model_loaded(x_test)

print("Difference:", torch.abs(y1 - y2).max().item())


Difference: 0.0


In [15]:
class MeanCenter(nn.Module):
    def forward(self, x):
        return x - x.mean(dim=1, keepdim=True)

class CustomMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            MeanCenter(),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x)

model_c = CustomMLP().to(device)
train(model_c, X, y)

Epoch 0 | Loss 4.3916
Epoch 1 | Loss 4.1065
Epoch 2 | Loss 3.8658
Epoch 3 | Loss 3.6003
Epoch 4 | Loss 3.2510


1. Model as a Composition of Modules

A neural network in PyTorch is a tree of nn.Module objects.

Each layer is a module that contains parameters and computation logic.


2. Parameter Management

Model parameters can be inspected via named_parameters() and state_dict().

We can directly access weights and gradients during training.


3. Parameter Sharing (Core Insight)

A shared layer uses the same weights (W, b) multiple times.

During backpropagation:

  Each usage contributes a gradient

  Final gradient = sum of all contributions


The parameter is updated once per step, using the accumulated gradient.


4. Backpropagation Mechanics

Gradients flow backward through the computational graph.

If a parameter appears in multiple paths, gradients from all paths are added:


Gradient of a variable = sum of gradients from all paths leading to it


5. Training Dynamics

Observing .grad helps understand how learning happens.

Shared layers often have larger gradients due to accumulation.


6. Saving and Loading Models

state_dict() stores all model parameters.

Models can be saved and restored accurately using this mechanism.


7. Custom Layers

New layers can be created by subclassing nn.Module.

This allows full flexibility in defining operations.